##  개인정보 제공 동의의 계층적 조건: 객관적 경제 조건과 주관적 경제 인식의 비교

### 선행연구


**조사필요**

### 가설

본 연구는 개인정보 제공 동의(`GIVEPIFR_R`)에 영향을 미치는 경제적 요인을 **객관적 경제 조건**과 **주관적 경제 인식**으로 구분한다. 객관적 경제 조건은 월평균 가구소득의 로그값인 `log_INCOM0`으로 측정하고, 주관적 경제 인식은 `RANK`, `SATFIN_R`, `FINPROS_R`로 측정한다.

`GIVEPIFR_R`은 원척도 `GIVEPIFR`을 역코딩한 변수이므로, 값이 높을수록 할인이나 무료상품을 받기 위해 개인정보를 제공하는 데 더 동의한다는 의미이다.

**가설 설정은 선행연구 살펴보고 정할 수 있을거 같습니다**

### 변수 

- 독립변수: 
- 원 종속변수: `GIVEPIFR`, 1=매우 동의, 5=매우 반대
- 분석 종속변수: `GIVEPIFR_R = 6 - GIVEPIFR`, 1=매우 반대, 5=매우 동의


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from IPython.display import display
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from patsy import dmatrices
from statsmodels.stats.outliers_influence import variance_inflation_factor

## 1. 데이터 불러오기

In [ ]:
DATA_PATH = Path(r"/home/pck/data_jour/sda_Team Project_/adt_2025_drop_0509.csv")
df_raw = pd.read_csv(DATA_PATH)

print(df_raw.shape)
display(df_raw.head())

## 2. 분석 변수 지정

분석에 사용할 핵심 변수를 지정한다. 

In [ ]:
raw_dv = "GIVEPIFR"
dv = "GIVEPIFR_R"

raw_iv = ["INCOM0", "RANK", "SATFIN", "FINPROS"]
iv = ["INCOM0", "RANK", "SATFIN_R", "FINPROS_R"]

raw_control = ["AGE", "SEX", "EDUC", "ABLITINF", "TECHHARM"]
control = ["AGE", "SEX", "EDUC_GROUP", "ABLITINF_R", "TECHHARM_R"]

weight = "FINALWT" # wls 

analysis_vars_raw = [raw_dv] + raw_iv + raw_control + [weight]

missing = [v for v in analysis_vars_raw if v not in df_raw.columns]
if missing:
    raise ValueError(f"데이터에 없는 변수: {missing}")

display(df_raw[analysis_vars_raw].describe().T)

## 3. 데이터 전처리

- `-8`은 공통 결측 코드로 처리한다.
- `ABLITINF`의 `-1`은 비해당으로 보고 결측 처리한다.
- `GIVEPIFR`, `SATFIN`, `FINPROS`, `ABLITINF`, `TECHHARM`은 분석 방향에 맞게 역코딩한다.
- `INCOM0`는 `log_INCOM0 = log(INCOM0 + 1)`로 변환한다.


### `FINALWT`와 WLS 사용

`FINALWT`는 KGSS에서 제공하는 최종 표본가중치이다. 표본조사에서는 조사 표본이 모집단의 성별, 연령, 지역 등 분포와 완전히 일치하지 않을 수 있으므로, 각 응답자가 모집단에서 어느 정도의 대표성을 갖는지를 보정하기 위해 가중치를 사용한다. 
> kgss 공식 홈페이지 설명: '모든 연도에 적용할 가중치 변수는 가구에서 적격한 성인가구원 추출할 확률과 성별/연령/지역/도시(농촌)을 고려하여 만든 FINALWT 입니다.' (링크: https://kgss.skku.edu/kgss/faq.do#a )

본 분석에서는 `FINALWT`를 가중치로 사용한 WLS(Weighted Least Squares)를 적용한다. WLS는 모든 응답자를 동일한 비중으로 취급하는 OLS와 달리, 가중치가 큰 응답자는 더 큰 비중으로, 가중치가 작은 응답자는 더 작은 비중으로 반영한다. 따라서 본 연구의 WLS 결과는 KGSS의 표본가중치를 반영한 회귀분석 결과로 해석한다.
> 즉, 2025년 표본 자체가 모집단을 완벽히 반영하는 표본이 아닐 수 있기 때문에, 표본의 모집단 대표성을 보정하는 측면에서 사용한다. 


In [ ]:
df = df_raw[analysis_vars_raw].copy()

# FINALWT는 이후 WLS 회귀분석에서 표본가중치로 사용한다.

# 공통 결측 코드 처리
for col in analysis_vars_raw:
    df[col] = df[col].replace(-8, np.nan)

# ABLITINF의 -1은 비해당으로 보고 결측 처리
df["ABLITINF"] = df["ABLITINF"].replace(-1, np.nan)

# 종속변수 개인정보 제공 태도 역코딩: 원척도 1=매우 동의, 5=매우 반대
# 분석척도는 1=매우 반대, 5=매우 동의
df["GIVEPIFR_R"] = np.where(df["GIVEPIFR"].isin([1, 2, 3, 4, 5]), 6 - df["GIVEPIFR"], np.nan)

# 독립변수/통제변수 역코딩
# SATFIN_R: 1=매우 불만족, 5=매우 만족
# FINPROS_R: 1=상당히 나빠질 것, 5=상당히 좋아질 것
# ABLITINF_R: 1=매우 낮음, 5=매우 높음
for v in ["SATFIN", "FINPROS", "ABLITINF"]:
    df[f"{v}_R"] = np.where(df[v].isin([1, 2, 3, 4, 5]), 6 - df[v], np.nan)

# TECHHARM_R: 값이 높을수록 기술위험 인식이 높도록 역코딩
df["TECHHARM_R"] = np.where(df["TECHHARM"].isin([1, 2, 3, 4, 5]), 6 - df["TECHHARM"], np.nan)

# INCOM0 변수 로그 변환
df["log_INCOM0"] = np.log1p(df["INCOM0"].where(df["INCOM0"] >= 0, np.nan))

# EDUC -> EDUC_GROUP
# 고졸 이하: 0=무학, 1=초등학교, 2=중학교, 3=고등학교, 8=서당한학 / 2년제 대학: 4=전문대학 / 4년제 대학: 5=대학교 / 대학원 이상: 6=대학원 석사, 7=대학원 박사
educ_categories = ["고졸 이하", "2년제 대학", "4년제 대학", "대학원 이상"]
df["EDUC_GROUP"] = pd.Series(pd.NA, index=df.index, dtype="object")
df.loc[df["EDUC"].isin([0, 1, 2, 3, 8]), "EDUC_GROUP"] = "고졸 이하"
df.loc[df["EDUC"].eq(4), "EDUC_GROUP"] = "2년제 대학"
df.loc[df["EDUC"].eq(5), "EDUC_GROUP"] = "4년제 대학"
df.loc[df["EDUC"].isin([6, 7]), "EDUC_GROUP"] = "대학원 이상"
df["EDUC_GROUP"] = pd.Categorical(df["EDUC_GROUP"], categories=educ_categories, ordered=True)


display(df.head())
display(df[["GIVEPIFR", "GIVEPIFR_R", "SATFIN_R", "FINPROS_R", "ABLITINF_R", "log_INCOM0", "EDUC", "EDUC_GROUP", "TECHHARM_R"]].head())
display(df.isna().sum())

In [ ]:
# NaN 값 제거

analysis_vars = [
    "GIVEPIFR_R",
    "log_INCOM0", "RANK", "SATFIN_R", "FINPROS_R",
    "AGE", "SEX", "EDUC_GROUP", "ABLITINF_R", "TECHHARM_R",
    "FINALWT",
]

adt = df.dropna(subset=analysis_vars).copy().reset_index(drop=True)

print(f"결측 제거 전 df 행 수: {len(df)}")
print(f"결측 제거 후 adt 행 수: {len(adt)}")
print(f"제거된 행 수: {len(df) - len(adt)}")

display(adt.head())
display(adt[analysis_vars].isna().sum())

## 4. 기술통계와 상관관계

In [ ]:
summary_vars = [
    "GIVEPIFR", "GIVEPIFR_R",
    "INCOM0", "log_INCOM0", "RANK",
    "SATFIN", "SATFIN_R", "FINPROS", "FINPROS_R",
    "AGE", "SEX", "EDUC", "ABLITINF", "ABLITINF_R", "TECHHARM", "TECHHARM_R",
]
display(adt[summary_vars].describe().T)

# 범주화한 학력 변수 분포
display(adt["EDUC_GROUP"].value_counts().reindex(educ_categories).to_frame("count"))
display((adt["EDUC_GROUP"].value_counts(normalize=True).reindex(educ_categories) * 100).round(2).to_frame("percent"))

corr_vars = ["GIVEPIFR_R", "INCOM0", "log_INCOM0", "RANK", "SATFIN_R", "FINPROS_R", "ABLITINF_R", "TECHHARM_R"]
corr_matrix = adt[corr_vars].corr(method="pearson")
display(corr_matrix)

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
)
plt.title("Pearson correlation")
plt.tight_layout()
plt.show()

## 5. 등분산성 검정

회귀 가정 점검용 Breusch-Pagan 검정과 White 검정.

In [ ]:
assumption_formula_main = (
    "GIVEPIFR_R ~ log_INCOM0 + RANK + SATFIN_R + FINPROS_R + AGE "
    "+ C(SEX) + C(EDUC_GROUP) + ABLITINF_R + TECHHARM_R"
)

# wresid는 statsmodels가 WLS 계산에 사용한 가중 잔차다.
# BP/White 검정 함수는 상수항이 있는 설명변수 행렬을 요구하므로 exog는 원 설계행렬을 사용한다.
assumption_wls = smf.wls(
    formula=assumption_formula_main,
    data=adt,
    weights=adt["FINALWT"]
).fit()

bp = het_breuschpagan(assumption_wls.wresid, assumption_wls.model.exog)
white = het_white(assumption_wls.wresid, assumption_wls.model.exog)

heteroskedasticity_results = pd.DataFrame([{
    "model": "ASSUMPTION_WLS_GIVEPIFR_R_logINCOM0_main",
    "nobs": int(assumption_wls.nobs),
    "bp_lm_stat": bp[0],
    "bp_lm_pvalue": bp[1],
    "bp_f_stat": bp[2],
    "bp_f_pvalue": bp[3],
    "white_lm_stat": white[0],
    "white_lm_pvalue": white[1],
    "white_f_stat": white[2],
    "white_f_pvalue": white[3],
    "bp_heteroskedasticity_5pct": bp[1] < 0.05,
    "white_heteroskedasticity_5pct": white[1] < 0.05,
}])

display(heteroskedasticity_results)


질문: BP/White 모두 5% 수준에서 이분산성이 있는 것으로 나온다. 강건표준오차를 사용해야 할까? (회귀계수는 그대로 두고, 표준오차와 p값만 이분산성에 덜 민감하게 계산)

혹은 
  - 주 분석: FINALWT를 반영한 WLS
  - 보조 확인: WLS + HC3 강건표준오차

  두 분석을 진행한 후 두 결과의 방향성과 유의성이 크게 달라지는지 비교?
  > 만약, HC3의 결과와 wls의 결과가 거의 같으면 결과의 신뢰성이 높아지며, 유의성이 크게 바뀐다면 wls의 p값 해석을 유의해야 한다. 

## 6. WLS 회귀분석: 분석 모형 구성

종속변수: `GIVEPIFR_R` (값이 높을수록 할인이나 무료상품을 받기 위해 개인정보를 제공하는 데 더 동의한다는 의미)

통제변수는 다음과 같다.

| 변수 | 분석상 역할 | 해석 |
|---|---|---|
| `AGE` | 통제변수 | 응답자 연령(min=18, max=93) |
| `SEX` | 통제변수 | 응답자 성별. 회귀식에서는 `C(SEX)`로 범주형 처리 |
| `EDUC_GROUP` | 통제변수 | 응답자 학력을 고졸 이하, 2년제 대학, 4년제 대학, 대학원 이상으로 재범주화한 변수. 회귀식에서는 `C(EDUC_GROUP)`으로 범주형 처리 |
| `ABLITINF_R` | 통제변수 | 인터넷 정보판별능력 역코딩 변수. 값이 높을수록 정보판별능력이 높음 |
| `TECHHARM_R` | 통제변수 | 기술위험 인식 역코딩 변수. 값이 높을수록 기술위험 인식이 높음 |

분석별 독립변수 구성은 다음과 같다.

| 분석 | 종속변수 | 독립변수 | 통제변수 |
|---|---|---|---|
| 분석 1 | `GIVEPIFR_R` | `log_INCOM0` | `AGE`, `SEX`, `EDUC_GROUP`, `ABLITINF_R`, `TECHHARM_R` |
| 분석 2 | `GIVEPIFR_R` | `RANK` | `AGE`, `SEX`, `EDUC_GROUP`, `ABLITINF_R`, `TECHHARM_R` |
| 분석 3 | `GIVEPIFR_R` | `SATFIN_R` | `AGE`, `SEX`, `EDUC_GROUP`, `ABLITINF_R`, `TECHHARM_R` |
| 분석 4 | `GIVEPIFR_R` | `FINPROS_R` | `AGE`, `SEX`, `EDUC_GROUP`, `ABLITINF_R`, `TECHHARM_R` |
| 분석 5 | `GIVEPIFR_R` | `RANK`, `SATFIN_R`, `FINPROS_R` | `AGE`, `SEX`, `EDUC_GROUP`, `ABLITINF_R`, `TECHHARM_R` |
| 분석 6 | `GIVEPIFR_R` | `log_INCOM0`, `RANK`, `SATFIN_R`, `FINPROS_R` | `AGE`, `SEX`, `EDUC_GROUP`, `ABLITINF_R`, `TECHHARM_R` |


In [ ]:
# 분석 1: log_INCOM0 + 통제변수
formula_analysis1 = (
    "GIVEPIFR_R ~ log_INCOM0 + AGE + C(SEX) + C(EDUC_GROUP) + ABLITINF_R + TECHHARM_R"
)
analysis1_model = smf.wls(
    formula=formula_analysis1,
    data=adt,
    weights=adt["FINALWT"]
).fit()

# 분석 2: RANK + 통제변수
formula_analysis2 = (
    "GIVEPIFR_R ~ RANK + AGE + C(SEX) + C(EDUC_GROUP) + ABLITINF_R + TECHHARM_R"
)
analysis2_model = smf.wls(
    formula=formula_analysis2,
    data=adt,
    weights=adt["FINALWT"]
).fit()

# 분석 3: SATFIN_R + 통제변수
formula_analysis3 = (
    "GIVEPIFR_R ~ SATFIN_R + AGE + C(SEX) + C(EDUC_GROUP) + ABLITINF_R + TECHHARM_R"
)
analysis3_model = smf.wls(
    formula=formula_analysis3,
    data=adt,
    weights=adt["FINALWT"]
).fit()

# 분석 4: FINPROS_R + 통제변수
formula_analysis4 = (
    "GIVEPIFR_R ~ FINPROS_R + AGE + C(SEX) + C(EDUC_GROUP) + ABLITINF_R + TECHHARM_R"
)
analysis4_model = smf.wls(
    formula=formula_analysis4,
    data=adt,
    weights=adt["FINALWT"]
).fit()

# 분석 5: RANK, SATFIN_R, FINPROS_R + 통제변수
formula_analysis5 = (
    "GIVEPIFR_R ~ RANK + SATFIN_R + FINPROS_R + AGE "
    "+ C(SEX) + C(EDUC_GROUP) + ABLITINF_R + TECHHARM_R"
)
analysis5_model = smf.wls(
    formula=formula_analysis5,
    data=adt,
    weights=adt["FINALWT"]
).fit()

# 분석 6: log_INCOM0, RANK, SATFIN_R, FINPROS_R + 통제변수
formula_analysis6 = (
    "GIVEPIFR_R ~ log_INCOM0 + RANK + SATFIN_R + FINPROS_R + AGE "
    "+ C(SEX) + C(EDUC_GROUP) + ABLITINF_R + TECHHARM_R"
)
analysis6_model = smf.wls(
    formula=formula_analysis6,
    data=adt,
    weights=adt["FINALWT"]
).fit()

models = {
    "ANALYSIS_1_WLS_GIVEPIFR_R_logINCOM0": analysis1_model,
    "ANALYSIS_2_WLS_GIVEPIFR_R_RANK": analysis2_model,
    "ANALYSIS_3_WLS_GIVEPIFR_R_SATFIN_R": analysis3_model,
    "ANALYSIS_4_WLS_GIVEPIFR_R_FINPROS_R": analysis4_model,
    "ANALYSIS_5_WLS_GIVEPIFR_R_SUBJECTIVE": analysis5_model,
    "ANALYSIS_6_WLS_GIVEPIFR_R_FULL": analysis6_model,
}

# 핵심 독립변수 중심 요약표
key_terms_by_model = {
    "ANALYSIS_1_WLS_GIVEPIFR_R_logINCOM0": ["log_INCOM0"],
    "ANALYSIS_2_WLS_GIVEPIFR_R_RANK": ["RANK"],
    "ANALYSIS_3_WLS_GIVEPIFR_R_SATFIN_R": ["SATFIN_R"],
    "ANALYSIS_4_WLS_GIVEPIFR_R_FINPROS_R": ["FINPROS_R"],
    "ANALYSIS_5_WLS_GIVEPIFR_R_SUBJECTIVE": ["RANK", "SATFIN_R", "FINPROS_R"],
    "ANALYSIS_6_WLS_GIVEPIFR_R_FULL": ["log_INCOM0", "RANK", "SATFIN_R", "FINPROS_R"],
}

result_rows = []
for model_name, model in models.items():
    conf_int = model.conf_int()
    for term in key_terms_by_model[model_name]:
        result_rows.append({
            "model": model_name,
            "term": term,
            "coef": model.params[term],
            "std_err": model.bse[term],
            "p_value": model.pvalues[term],
            "ci_low": conf_int.loc[term, 0],
            "ci_high": conf_int.loc[term, 1],
            "nobs": int(model.nobs),
            "r_squared": model.rsquared,
            "adj_r_squared": model.rsquared_adj,
        })

wls_key_results = pd.DataFrame(result_rows)
display(wls_key_results)

# 전체 회귀표
for model_name, model in models.items():
    print("=" * 100)
    print(model_name)
    print(model.summary())


## 7. 분석결과

*아래 해석은 모든 분석에서 `AGE`, `SEX`, `EDUC_GROUP`, `ABLITINF_R`, `TECHHARM_R`를 통제한 WLS 결과를 기준으로 한다.*


> **분석 1:** `log_INCOM0`의 계수는 -0.029이고 p값은 0.564로 통계적으로 유의하지 않다. 즉, 객관적 경제 조건인 월평균 가구소득은 개인정보 제공 동의와 뚜렷한 관련을 보이지 않는다.

> **분석 2:** `RANK`의 계수는 0.023이고 p값은 0.323으로 통계적으로 유의하지 않다. 주관적 계층의식을 단독으로 투입했을 때는 개인정보 제공 동의와 유의한 관련이 확인되지 않는다.

> **분석 3:** `SATFIN_R`의 계수는 -0.077이고 p값은 0.047로 5% 수준에서 유의하다. `SATFIN_R`은 값이 높을수록 가계상태 만족도가 높다는 뜻이므로, 가계상태 만족도가 높을수록 개인정보 제공 동의는 낮아지는 방향이다. 반대로 가계상태 만족도가 낮을수록 개인정보 제공 동의가 높다고 해석할 수 있다.

> **분석 4:** `FINPROS_R`의 계수는 -0.050이고 p값은 0.286으로 통계적으로 유의하지 않다. 가계경제 전망은 단독 모형에서 개인정보 제공 동의와 뚜렷한 관련을 보이지 않는다.

> **분석 5:** 주관적 경제 인식 변수인 `RANK`, `SATFIN_R`, `FINPROS_R`를 함께 투입한 결과, `RANK`는 양의 방향으로 유의하고(coef=0.055, p=0.037), `SATFIN_R`은 음의 방향으로 유의하다(coef=-0.108, p=0.012). `FINPROS_R`는 유의하지 않다(p=0.433). 즉, 주관적 계층의식이 높을수록 개인정보 제공 동의가 높고, 가계상태 만족도가 낮을수록 개인정보 제공 동의가 높은 경향이 나타난다.

> **분석 6:** 객관적 경제 조건(`log_INCOM0`)과 주관적 경제 인식(`RANK`, `SATFIN_R`, `FINPROS_R`)을 함께 투입한 결과, `log_INCOM0`는 유의하지 않다(p=0.511). 반면 `RANK`는 양의 방향으로 유의하고(coef=0.060, p=0.029), `SATFIN_R`은 음의 방향으로 유의하다(coef=-0.105, p=0.016). `FINPROS_R`는 유의하지 않다(p=0.442). 따라서 객관적 소득보다는 주관적 계층의식과 현재 가계상태 만족도가 개인정보 제공 동의와 더 관련되어 있는 것으로 보인다.

>> 전체적으로 설명력은 R-squared 기준 약 0.054~0.062 이다.   
>> 분석 5와 분석 6에서 `RANK`와 `SATFIN_R`의 방향과 유의성이 비교적 일관되게 나타난다.


## 8. 다중공선성 점검: VIF

In [ ]:
vif_formula_main = formula_analysis6


def calculate_vif(formula, data, model_name):
    y, X = dmatrices(formula, data=data, return_type="dataframe")
    rows = []
    for i, col in enumerate(X.columns):
        if col == "Intercept":
            continue
        rows.append({
            "model": model_name,
            "term": col,
            "vif": variance_inflation_factor(X.values, i),
        })
    return pd.DataFrame(rows).sort_values("vif", ascending=False).reset_index(drop=True)

vif_results = calculate_vif(vif_formula_main, adt, "VIF_analysis6_full_model")
display(vif_results)


## 9. 참고사항

- 모든 WLS 분석의 종속변수는 `GIVEPIFR_R`이다. 값이 높을수록 할인/무료상품을 위한 개인정보 제공 동의가 높다는 의미이다.
- WLS 회귀에서 계수가 양수이면 해당 독립변수 또는 통제변수가 증가할수록 개인정보 제공 동의 수준이 높아지는 방향이며, 계수가 음수이면 해당 독립변수 또는 통제변수가 증가할수록 개인정보 제공 동의 수준이 낮아지는 방향이다.
- 분석 1은 **객관적 경제 조건**인 `log_INCOM0`의 관계를 확인한다.
- 분석 2는 주관적 계층의식 `RANK`의 관계를 확인한다.
- 분석 3은 가계상태 만족도 `SATFIN_R`의 관계를 확인한다.
- 분석 4는 가계경제 전망 `FINPROS_R`의 관계를 확인한다.
- 분석 5는 **주관적 경제인식**인 `RANK`, `SATFIN_R`, `FINPROS_R`를 동시에 투입하여 '경제적 위치와 상황에 대한 개인의 인식과 평가'를 살펴본다. 
- 분석 6은 `log_INCOM0`, `RANK`, `SATFIN_R`, `FINPROS_R`를 동시에 투입해 객관적 소득까지 포함한 종합 관계를 확인한다.
- `SATFIN_R`, `FINPROS_R`, `ABLITINF_R`는 모두 값이 높을수록 각각 가계상태 만족, 가계경제 전망 긍정, 정보판별능력이 높다는 뜻이다.
- `TECHHARM_R`는 값이 높을수록 기술위험 인식이 높다는 뜻이다.  
- `EDUC_GROUP`은 원 학력 변수 `EDUC`를 고졸 이하, 2년제 대학, 4년제 대학, 대학원 이상으로 재범주화한 변수이며 회귀식에서는 범주형으로 처리한다.
